# Week 2 — Gaussian Process optimisation

Update the models using the observations available at the start of Week 2 and select the next query for each function.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

working_dir = Path.cwd()
possible_data_folders = [working_dir.parent / "Data", working_dir / "Data"]
data_folder = next((p for p in possible_data_folders if all((p / f"function_{i}.csv").is_file() for i in range(1, 9))), None)
if data_folder is None:
    raise FileNotFoundError(f"Cannot find function_1.csv through function_8.csv in a Data folder near {working_dir}")
print("Loading Week 2 observations from:", data_folder)


Loading Week 2 observations from: C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\2. Module 13\Data


## 1. Load the cumulative Week 2 observations


In [2]:
data = {}
for i in range(1, 9):
    df = pd.read_csv(data_folder / f"function_{i}.csv")
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed:")]
    df.columns = df.columns.str.strip()
    if "output" not in df.columns:
        raise ValueError(f"function_{i}.csv needs an 'output' column; found {list(df.columns)}")
    input_cols = [c for c in df.columns if c != "output"]
    expected = [f"x{j+1}" for j in range(len(input_cols))]
    if set(input_cols) != set(expected):
        raise ValueError(f"function_{i}.csv has unexpected input columns: {input_cols}")
    df = df[expected + ["output"]].apply(pd.to_numeric, errors="raise")
    data[i] = df
    print(f"Function {i}: {len(df)} observations, {len(expected)} dimensions")


Function 1: 11 observations, 2 dimensions
Function 2: 11 observations, 2 dimensions
Function 3: 16 observations, 3 dimensions
Function 4: 31 observations, 4 dimensions
Function 5: 21 observations, 4 dimensions
Function 6: 21 observations, 5 dimensions
Function 7: 31 observations, 6 dimensions
Function 8: 41 observations, 8 dimensions


## 2. Compare the returned observations with previous best outputs


In [3]:
for i in range(1, 9):
    df = data[i]
    if len(df) < 2:
        raise ValueError(f"Function {i} needs at least two observations")
    new_output = df.iloc[-1]["output"]
    previous_best = df.iloc[:-1]["output"].max()
    updated_best = df["output"].max()
    print(f"Function {i}: previous best={previous_best:.6f}, new={new_output:.6f}, "
          f"difference={new_output-previous_best:+.6f}, updated best={updated_best:.6f}, "
          f"new best={bool(new_output > previous_best)}")


Function 1: previous best=0.000000, new=0.000000, difference=-0.000000, updated best=0.000000, new best=False
Function 2: previous best=0.611205, new=-0.082041, difference=-0.693246, updated best=0.611205, new best=False
Function 3: previous best=-0.034835, new=-0.020167, difference=+0.014668, updated best=-0.020167, new best=True
Function 4: previous best=-4.025542, new=-0.300645, difference=+3.724898, updated best=-0.300645, new best=True
Function 5: previous best=1088.859618, new=2329.015854, difference=+1240.156236, updated best=2329.015854, new best=True
Function 6: previous best=-0.714265, new=-2.661556, difference=-1.947291, updated best=-0.714265, new best=False
Function 7: previous best=1.364968, new=1.950612, difference=+0.585644, updated best=1.950612, new best=True
Function 8: previous best=9.598482, new=9.821986, difference=+0.223504, updated best=9.821986, new best=True


## 3. Refit the RBF Gaussian Processes


In [4]:
models_week2 = {}

for i in range(1, 9):

    df = data[i]

    X = df.drop(columns="output").values
    y = df["output"].values

    kernel = RBF(
        length_scale=0.5,
        length_scale_bounds=(1e-5, 1e5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)

    models_week2[i] = gp

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print("Fitted kernel:", gp.kernel_)
    print()


FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0453)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.202)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.522)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.225)

FUNCTION 6
Fitted kernel: RBF(length_scale=0.481)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.354)

FUNCTION 8
Fitted kernel: RBF(length_scale=0.913)



## 4. Compare RBF and Matérn for Function 1


In [5]:
from sklearn.gaussian_process.kernels import Matern

df = data[1]

X = df.drop(columns="output").values
y = df["output"].values

rbf_kernel = RBF(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5)
)

rbf_gp = GaussianProcessRegressor(
    kernel=rbf_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

rbf_gp.fit(X, y)


matern_kernel = Matern(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5),
    nu=2.5
)

matern_gp = GaussianProcessRegressor(
    kernel=matern_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

matern_gp.fit(X, y)


print("RBF kernel:")
print(rbf_gp.kernel_)

print("\nMatérn kernel:")
print(matern_gp.kernel_)


RBF kernel:
RBF(length_scale=1e-05)

Matérn kernel:
Matern(length_scale=1e-05, nu=2.5)


## 5. Compare the predictions for Function 1


In [6]:
rng = np.random.default_rng(42)

X_candidates_f1 = rng.uniform(
    low=0,
    high=1,
    size=(100000, 2)
)

rbf_mean, rbf_std = rbf_gp.predict(
    X_candidates_f1,
    return_std=True
)

matern_mean, matern_std = matern_gp.predict(
    X_candidates_f1,
    return_std=True
)

print("=" * 60)
print("FUNCTION 1 - RBF vs MATERN")
print("=" * 60)

print("\nRBF")
print(f"Predicted output range: {rbf_mean.min():.10f} to {rbf_mean.max():.10f}")
print(f"Uncertainty range:      {rbf_std.min():.10f} to {rbf_std.max():.10f}")

print("\nMATERN")
print(f"Predicted output range: {matern_mean.min():.10f} to {matern_mean.max():.10f}")
print(f"Uncertainty range:      {matern_std.min():.10f} to {matern_std.max():.10f}")


FUNCTION 1 - RBF vs MATERN

RBF
Predicted output range: -0.0003278239 to -0.0000001931
Uncertainty range:      0.0000355799 to 0.0010366701

MATERN
Predicted output range: -0.0003278239 to -0.0000003215
Uncertainty range:      0.0000459032 to 0.0010366701


## 6. Generate candidate points and apply UCB


In [7]:
week2_candidates = {}

rng = np.random.default_rng(42)

kappa = 2.5

for i in range(2, 9):

    df = data[i]
    gp = models_week2[i]

    n_dimensions = len(df.columns) - 1

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    ucb = mean + kappa * std

    best_index = np.argmax(ucb)
    selected_point = X_candidates[best_index]

    week2_candidates[i] = selected_point

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:", selected_point)
    print("Predicted output:", mean[best_index])
    print("Uncertainty:", std[best_index])
    print("UCB:", ucb[best_index])
    print()


FUNCTION 2
Selected point: [0.65528565 0.92989635]
Predicted output: 0.44107070745260113
Uncertainty: 0.1900322877790721
UCB: 0.9161514269002814

FUNCTION 3
Selected point: [0.39752993 0.09047714 0.44731895]
Predicted output: -0.055801135420240754
Uncertainty: 0.07430259032309243
UCB: 0.12995534038749032

FUNCTION 4
Selected point: [0.48075601 0.47400033 0.22905919 0.42917843]
Predicted output: -1.6665064041368183
Uncertainty: 0.8868534178318094
UCB: 0.550627140442705

FUNCTION 5
Selected point: [0.51484244 0.85610509 0.96150603 0.99849219]
Predicted output: 2112.9037956014745
Uncertainty: 303.2913666553353
UCB: 2871.132212239813

FUNCTION 6
Selected point: [0.24654859 0.34464495 0.47952502 0.733976   0.02533225]
Predicted output: -0.6645463641633733
Uncertainty: 0.3554831146702004
UCB: 0.22416142251212767

FUNCTION 7
Selected point: [0.08033361 0.31746516 0.48551741 0.0555686  0.40688258 0.93197557]
Predicted output: 1.7729655691895485
Uncertainty: 0.24096833783176175
UCB: 2.375386413

## 7. Compare proposed queries with current best observations


In [8]:
for i in range(2, 9):

    df = data[i]

    best_index = df["output"].idxmax()
    best_row = df.loc[best_index]

    best_X = best_row.drop("output").values.astype(float)
    best_y = best_row["output"]

    new_X = week2_candidates[i]

    distance = np.linalg.norm(new_X - best_X)

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Current best point:")
    print(np.round(best_X, 6))

    print("Current best output:")
    print(round(best_y, 6))

    print("\nProposed new point:")
    print(np.round(new_X, 6))

    print("\nDistance from current best:")
    print(round(distance, 4))

    print()


FUNCTION 2
Current best point:
[0.702637 0.926564]
Current best output:
0.611205

Proposed new point:
[0.655286 0.929896]

Distance from current best:
0.0475

FUNCTION 3
Current best point:
[0.409499 0.399    0.519487]
Current best output:
-0.020167

Proposed new point:
[0.39753  0.090477 0.447319]

Distance from current best:
0.3171

FUNCTION 4
Current best point:
[0.399157 0.433203 0.357699 0.461233]
Current best output:
-0.300645

Proposed new point:
[0.480756 0.474    0.229059 0.429178]

Distance from current best:
0.1609

FUNCTION 5
Current best point:
[0.352268 0.854957 0.92921  0.999541]
Current best output:
2329.015854

Proposed new point:
[0.514842 0.856105 0.961506 0.998492]

Distance from current best:
0.1658

FUNCTION 6
Current best point:
[0.728186 0.154693 0.732552 0.693997 0.056401]
Current best output:
-0.714265

Proposed new point:
[0.246549 0.344645 0.479525 0.733976 0.025332]

Distance from current best:
0.5785

FUNCTION 7
Current best point:
[0.120627 0.403921 0.379

## 8. Select an exploration point for Function 1


In [9]:
from scipy.spatial.distance import cdist

X_existing = data[1].drop(columns="output").values

X_candidates = X_candidates_f1

distances = cdist(X_candidates, X_existing)

nearest_distance = distances.min(axis=1)

best_index = np.argmax(nearest_distance)

function1_point = X_candidates[best_index]
function1_distance = nearest_distance[best_index]

print("=" * 60)
print("FUNCTION 1 - EXPLORATION")
print("=" * 60)

print("Selected point:")
print(function1_point)

print("\nDistance to nearest existing observation:")
print(function1_distance)


FUNCTION 1 - EXPLORATION
Selected point:
[0.99997255 0.99984253]

Distance to nearest existing observation:
0.3456413874867929


## 9. Combine the final query points


In [10]:

final_points = {
    1: function1_point,
    2: week2_candidates[2],
    3: week2_candidates[3],
    4: week2_candidates[4],
    5: week2_candidates[5],
    6: week2_candidates[6],
    7: week2_candidates[7],
    8: week2_candidates[8]
}


## 10. Validate the final queries


In [11]:
for i in range(1, 9):

    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values

    in_bounds = np.all((point >= 0) & (point <= 1))

    distances = np.linalg.norm(X_existing - point, axis=1)

    min_distance = distances.min()

    duplicate = np.any(np.all(np.isclose(X_existing, point), axis=1))

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:")
    print(np.round(point, 6))

    print(f"In bounds: {in_bounds}")
    print(f"Duplicate existing point: {duplicate}")
    print(f"Distance to nearest observation: {min_distance:.6f}")

    if i == 1:
        expected_index = np.argmax(nearest_distance)
        selection_correct = np.allclose(
            point,
            X_candidates_f1[expected_index]
        )
        print(f"Maximum-distance selection correct: {selection_correct}")

    else:
        gp = models_week2[i]

        print("Selection method: UCB")

    print()


FUNCTION 1
Selected point:
[0.999973 0.999843]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.345641
Maximum-distance selection correct: True

FUNCTION 2
Selected point:
[0.655286 0.929896]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.047468
Selection method: UCB

FUNCTION 3
Selected point:
[0.39753  0.090477 0.447319]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.291689
Selection method: UCB

FUNCTION 4
Selected point:
[0.480756 0.474    0.229059 0.429178]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.160930
Selection method: UCB

FUNCTION 5
Selected point:
[0.514842 0.856105 0.961506 0.998492]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.165759
Selection method: UCB

FUNCTION 6
Selected point:
[0.246549 0.344645 0.479525 0.733976 0.025332]
In bounds: True
Duplicate existing point: False
Distance to ne

## 11. Display the final query points


In [12]:
queries = pd.DataFrame([
    {"function": i, "method": "max-distance" if i == 1 else "UCB", "point": " ".join(f"{v:.8f}" for v in final_points[i])}
    for i in range(1, 9)
])
print(queries.to_string(index=False))


 function       method                                                                                   point
        1 max-distance                                                                   0.99997255 0.99984253
        2          UCB                                                                   0.65528565 0.92989635
        3          UCB                                                        0.39752993 0.09047714 0.44731895
        4          UCB                                             0.48075601 0.47400033 0.22905919 0.42917843
        5          UCB                                             0.51484244 0.85610509 0.96150603 0.99849219
        6          UCB                                  0.24654859 0.34464495 0.47952502 0.73397600 0.02533225
        7          UCB                       0.08033361 0.31746516 0.48551741 0.05556860 0.40688258 0.93197557
        8          UCB 0.07812669 0.66871085 0.05418798 0.20948756 0.86355186 0.71779870 0.07443940 0.86629963
